Code for analyzing precipitation from CM2.5-FLOR Preindustrial simulations forced with CTRL and modified topographic boundary conditions and comparing model output to a suite of observational and reanalysis products.

Relevant for manuscript Figure 3 and Supplemental Figure S3

In [ ]:
import os
import sys
# block warnings from printing
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
from scipy.ndimage import gaussian_filter1d
import pyproj

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.ops import transform
from shapely.geometry import LineString

import matplotlib as mpl
from matplotlib.lines import Line2D
import matplotlib.patches as patches
from matplotlib.patches import Polygon
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib import cm

# settings
%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *
from domain_funcs import *

dpath0='/discover/nobackup/projects/giss/baldwin_nip/dmkumar' # top level data directory

## LOAD DATA

In [ ]:
### +++ DATA PATHS +++ ###

# product keys
keys=['obs','flor']
obs_prods=['imerg', 'trmm', 'gpcp', 'gpcc']
flor_runs=['ctrl','hicam','hitopo']

# store file paths in dictionary
files={}
    
for key in ['obs']:
    files[key] = {}
    files[key]['imerg'] = f'{dpath0}/obs_data/prec/imerg.gn.timeseries.2001-2018.nc' 
    files[key]['trmm'] = f'{dpath0}/obs_data/prec/pr_TRMM-L3_v7-7A_199801-201312.nc' 
    #files[key]['chirps'] = f'{dpath0}/obs_data/prec/chirps-v3.0.monthly.nc'
    files[key]['gpcc'] = f'{dpath0}/obs_data/prec/gpcc.precip.mon.rate.0.25x0.25.v2020.nc'
    files[key]['gpcp'] = f'{dpath0}/obs_data/prec/gpcp.precip.1979-2018.monthly.nc'

for key in ['flor']:
    files[key] = {}
    varn='precip'
    for run in flor_runs:
        files[key][run] = f'{dpath0}/FLOR/{run}/pi/flor.{run}.{varn}.monthly.nc'

for key in ['topo']:
    files[key] = {}
    files[key]['etopo'] = f'{dpath0}/topo_files/obs.etopo5.zsurf.nc'
    files[key]['ctrl'] = f'{dpath0}/topo_files/flor.ctrl.zsurf.nc'
    files[key]['hicam'] = f'{dpath0}/topo_files/flor.hicam.zsurf.nc'
    files[key]['hitopo'] = f'{dpath0}/topo_files/flor.hitopo.zsurf.nc'

In [ ]:
### +++ ORGANIZE DATA +++ ###

# lat lon bounds
latmin = 10
latmax = 70
lonmin = 200
lonmax = 300
# time bounds
n_years=100
t_idx = -1 * n_years * 365 # number of time steps to keep in days
n_keep=-1*(n_years*12) # number of time steps to keep in months

# initialize dictionaries
dat = { 'obs': {},
        'flor': {} }

print('Working on...')
for key in ['obs']:
    print(f'{key}')
    for run in obs_prods:
        if run == 'imerg':
            ds = xr.open_dataset(files[key][run], chunks={}).precipitation * 24 # convert from mm/hr to mm/day
            ds = ds.transpose('time','lat','lon')
            ds.attrs['units'] = 'mm/day'
            ds.attrs['Units'] = 'mm/day'
            ds = lonFlip(ds) #longitude_flip(ds) # switch lons from -180:180 to 0:360
        if run == 'trmm':
            ds = xr.open_dataset(files[key][run], chunks={}).pr * 86400 # convert from kg m-2 s-1 to mm/day
        if run == 'gpcp':
            ds = xr.open_dataset(files[key][run], chunks={}).precip
        if run == 'gpcc':
            ds = xr.open_dataset(files[key][run], chunks={}).precip.sortby("lat")  # sort lats from south to north
        dat[key][run] = ds.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
        del ds
        
for key in ['flor']:
    print(f'{key}')
    for run in flor_runs:
        # open diag files
        ds = xr.open_dataset(files[key][run]).precip[n_keep:,:,:] * 86400 # keep only last 50 years & convert from mm/s to mm/day
        ds = ds.rename({'grid_xt':'lon','grid_yt':'lat'}) # update coordinate names to match imerg
        ds.attrs['units'] = 'mm/day' # update units
        dat[key][run] = ds.sel(lon=slice(lonmin,lonmax), lat=slice(latmin,latmax))
    
print('Done.')

topo = {}
for key in ['topo']:
    for case in ['etopo']:
        ds = xr.open_dataset(files[key][case]).ROSE.rename({'ETOPO05_X':'lon', 'ETOPO05_Y':'lat'})
        topo[case] = ds.where(ds>0, np.nan).sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
        del ds
    for case in ['ctrl', 'hicam', 'hitopo']:
        ds = xr.open_dataset(files[key][case]).ZSURF.rename({'GRID_XT':'lon', 'GRID_YT':'lat'})
        topo[case] = ds.where(ds>0, np.nan).sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
        del ds

## CALCULATE TIME-MEANS
Focusing only on July-August-September because that is the peak North American Monsoon season.

In [ ]:
### +++ CALCULATE TIME-MEANS +++ ###
season='JAS'
mons=[7,8,9]
jas_mean = { 'obs': {},
            'flor': {} }
ann_jas_mean = { 'obs': {},
                 'flor': {} }

print('Calculating seasonal means for...')
for key in ['obs']:
    print(f'{key}')
    for run in obs_prods:
        # seasonal mean for whole timeseries
        custom_seasons = xr.where(dat[key][run]['time'].dt.month.isin(mons), season, 'Other')
        jas_mean[key][run] = dat[key][run].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
        # seasonal mean by year
        ann_jas_mean[key][run] = dat[key][run].sel(time=dat[key][run]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')
        
for key in ['flor']:
    print(f'{key}')
    for run in flor_runs:
        # seasonal mean for whole timeseries
        custom_seasons = xr.where(dat[key][run]['time'].dt.month.isin(mons), season, 'Other')
        jas_mean[key][run] = dat[key][run].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
        # seasonal mean by year
        ann_jas_mean[key][run] = dat[key][run].sel(time=dat[key][run]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')

print('Done.')

## SIGNIFICANCE TESTING

In [ ]:
### +++ COMPARING ALL MODEL RUNS TO OBS. +++ ###

## initialize dictionaries for re-gridded obs data
jas_mean_regrid = { 'imerg_flor':{}, 'trmm_flor':{}, 'gpcc_flor':{}, 'gpcp_flor':{} }
ann_jas_mean_regrid = { 'imerg_flor':{}, 'trmm_flor':{}, 'gpcc_flor':{}, 'gpcp_flor':{} }

# First, need to put obs data on same grid as model output
regrid_keys = ['imerg_flor','trmm_flor','gpcc_flor','gpcp_flor']

print('Re-gridding obs.')
for i,key in enumerate(obs_prods):
    lats=dat['flor']['ctrl'].lat
    lons=dat['flor']['ctrl'].lon
    regrid_key=regrid_keys[i]
    # interpolate obs to CM2.5-FLOR grid
    jas_mean_regrid[regrid_key] = jas_mean['obs'][key].interp(lat=lats, lon=lons, method='linear')
    ann_jas_mean_regrid[regrid_key] = ann_jas_mean['obs'][key].interp(lat=lats, lon=lons, method='linear')
print('Done.')

# Determine statistical significance of model-obs differences based on students t-test
imerg_diff      = { 'flor' : {} }
imerg_diff_mask = { 'flor' : {} }
imerg_ptvals    = { 'flor' : {} }

trmm_diff      = { 'flor' : {} }
trmm_diff_mask = { 'flor' : {} }
trmm_ptvals    = { 'flor' : {} }

gpcp_diff      = { 'flor' : {} }
gpcp_diff_mask = { 'flor' : {} }
gpcp_ptvals    = { 'flor' : {} }

gpcc_diff      = { 'flor' : {} }
gpcc_diff_mask = { 'flor' : {} }
gpcc_ptvals    = { 'flor' : {} }

print('\nSignificance testing for:')
for key in imerg_diff.keys():
    print(f'imerg\n{key}')
    for run in flor_runs:
        print(f'...{run}')
        diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[key][run], ann_jas_mean_regrid['imerg_flor'],
                                               jas_mean[key][run], jas_mean_regrid['imerg_flor'])
        imerg_diff[key][run] = diff_
        imerg_diff_mask[key][run] = diff_mask_
        imerg_ptvals[key][run] = ptvals_

for key in trmm_diff.keys():
    print(f'\ntrmm\n{key}')
    for run in flor_runs:
        print(f'...{run}')
        diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[key][run], ann_jas_mean_regrid['trmm_flor'],
                                               jas_mean[key][run], jas_mean_regrid['trmm_flor'])
        trmm_diff[key][run] = diff_
        trmm_diff_mask[key][run] = diff_mask_
        trmm_ptvals[key][run] = ptvals_

for key in gpcc_diff.keys():
    print(f'\ngpcc\n{key}')
    for run in flor_runs:
        print(f'...{run}')
        diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[key][run], ann_jas_mean_regrid['gpcc_flor'],
                                               jas_mean[key][run], jas_mean_regrid['gpcc_flor'])
        gpcc_diff[key][run] = diff_
        gpcc_diff_mask[key][run] = diff_mask_
        gpcc_ptvals[key][run] = ptvals_

for key in gpcp_diff.keys():
    print(f'\ngpcc\n{key}')
    for run in flor_runs:
        print(f'...{run}')
        diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[key][run], ann_jas_mean_regrid['gpcp_flor'],
                                               jas_mean[key][run], jas_mean_regrid['gpcp_flor'])
        gpcp_diff[key][run] = diff_
        gpcp_diff_mask[key][run] = diff_mask_
        gpcp_ptvals[key][run] = ptvals_

print('Done.')

In [ ]:
### +++ COMPARING ALL MODEL RUNS TO MODEL CTRL +++ ###

# initialize dictionaries
model_diff      = { 'flor' : {} }
model_diff_mask = { 'flor' : {} }
model_ptvals    = { 'flor' : {} }

# names of modified topography runs
flor_mod_runs=['hicam','hitopo']

print('Significance testing for:')
for key in model_diff.keys():
    print(f'{key}')
    for run in flor_mod_runs:
        print(f'...{run}')
        # calculate significance of model - obs difference
        diff_, diff_mask_, ptvals_ = sigtest(ann_jas_mean[key][run], ann_jas_mean[key]['ctrl'],
                                             jas_mean[key][run], jas_mean[key]['ctrl'])
        model_diff[key][run] = diff_
        model_diff_mask[key][run] = diff_mask_
        model_ptvals[key][run] = ptvals_

print('Done.')

In [ ]:
### +++ BIAS IMPROVEMENT +++ ###

imerg_bias_change        = { 'flor' : {} }
imerg_bias_change_masked = { 'flor' : {} }
trmm_bias_change        = { 'flor' : {} }
trmm_bias_change_masked = { 'flor' : {} }
gpcp_bias_change        = { 'flor' : {} }
gpcp_bias_change_masked = { 'flor' : {} }
gpcc_bias_change        = { 'flor' : {} }
gpcc_bias_change_masked = { 'flor' : {} }

# calculate difference in the absolute value of the model-obs precip difference
# to determine whether or not the change in precipitation is a reduction in the ctrl model bias
for key in imerg_bias_change.keys():
    for run in flor_mod_runs:
        imerg_bias_change[key][run] = np.abs(imerg_diff[key][run])-np.abs(imerg_diff[key]['ctrl'])
        imerg_bias_change_masked[key][run] = imerg_bias_change[key][run].where(model_diff_mask[key][run].mask==False,np.nan)

for key in trmm_bias_change.keys():
    for run in flor_mod_runs:
        trmm_bias_change[key][run] = np.abs(trmm_diff[key][run])-np.abs(trmm_diff[key]['ctrl'])
        trmm_bias_change_masked[key][run] = trmm_bias_change[key][run].where(model_diff_mask[key][run].mask==False,np.nan)

for key in gpcp_bias_change.keys():
    for run in flor_mod_runs:
        gpcp_bias_change[key][run] = np.abs(gpcp_diff[key][run])-np.abs(gpcp_diff[key]['ctrl'])
        gpcp_bias_change_masked[key][run] = gpcp_bias_change[key][run].where(model_diff_mask[key][run].mask==False,np.nan)

for key in gpcc_bias_change.keys():
    for run in flor_mod_runs:
        gpcc_bias_change[key][run] = np.abs(gpcc_diff[key][run])-np.abs(gpcc_diff[key]['ctrl'])
        gpcc_bias_change_masked[key][run] = gpcc_bias_change[key][run].where(model_diff_mask[key][run].mask==False,np.nan)

## FIGURES

In [ ]:
coords_map, regions = nam_regions()
domains = list(coords_map.keys())

### Obs Products JAS Climatologies

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['IMERG', 'TRMM', 'GPCP', 'GPCC'])
letters=['A','B','C','D']
tx=-110.5
ty=40.25
# bias colormap
dcmap,_,_,_=get_settings(field='precip', diff=False)
dvmin=0
dvmax=11
dlevels=np.linspace(dvmin, dvmax, 23)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[239,260,15,40]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(25,8), layout='constrained', subplot_kw={'projection':proj})

for i,prod in enumerate(['imerg','trmm','gpcp','gpcc']):
    obs_data = jas_mean['obs'][prod]
    lats = obs_data.lat
    lons = obs_data.lon
    cf = ax[i].pcolormesh(lons, lats, obs_data, cmap=dcmap, norm=dnorm, transform=trans)

for i, ax in enumerate(ax.flat): 
    """
    # add box around core NAM domain
    lon_bnds = np.array([-113, -104, -104, -113])
    lat_bnds = np.array([20,  20,  34,  34])
    ring=LinearRing(list(zip(lon_bnds, lat_bnds)))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=2, linestyle='--', zorder=11) 
    """
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-121, ty+0.5, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=1)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=1)
    ax.add_feature(cfeature.OCEAN, facecolor='grey', linewidth=1)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=(i==0); gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.02, 0.7])
cbar=fig.colorbar(cf, ticks=[1,3,5,7,9,11], orientation='vertical', extend='both', cax=cax)
cbar.set_label('Precipitation Rate [mm/day]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

plt.savefig('../figs/prec_climo_nam_observations.pdf', transparent=False, bbox_inches='tight')
plt.savefig('../figs/prec_climo_nam_observations.png', transparent=False, bbox_inches='tight')

### Prec Bias & Change vs. IMERG

In [ ]:
# --- Settings --- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['CTRL$-$IMERG', r'HI$_{\mathbf{gbl}}$$-$CTRL', r'HI$_{\mathbf{mex}}$$-$CTRL'])
letters=['A','B','C','D']
tx=-110.5
ty=40.25
# var specs
lat=jas_mean['flor']['ctrl'].lat
lon=jas_mean['flor']['ctrl'].lon
# bias colormap
dcmap,_,_,_=get_settings(field='precip', diff=True)
dvmin=-3
dvmax=3
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(1000,3000,7)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[239,260,15,40]


fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20,9), layout='constrained', subplot_kw={'projection':proj})

for key in ['flor']:
    # ctrl - imerg difference (i.e., model bias)
    cf=ax[0].pcolormesh(lon, lat, imerg_diff_mask[key]['ctrl'], cmap=dcmap, norm=dnorm, transform=trans)
    # add topo contours
    ax[0].contour(lon, lat, topo['ctrl'], levels=zlevels, linewidths=1.3, colors='black', transform=trans)

    for i,run in enumerate(['hitopo','hicam']):
        # modified_flor - ctrl difference (i.e., impact of topography)
        ax[i+1].pcolormesh(lon, lat, model_diff_mask[key][run], cmap=dcmap, norm=dnorm, transform=trans)
        # add topo contours
        ax[i+1].contour(lon, lat, topo[run], levels=zlevels, linewidths=1.3, colors='black', transform=trans)
        # add hatching where bias got worse
        ax[i+1].contourf(lon, lat, imerg_bias_change_masked[key][run],
                       0, colors='none', hatches=['','///'], extend='lower', zorder=100, transform=trans)
        

for i, ax in enumerate(ax.flat): 
    # add boxes around NAM sub-domains
    poly1 = patches.Polygon(
        coords_map['north'],
        closed=True, ec='firebrick', fc='none', lw=2.5, ls='--',
        transform=ccrs.PlateCarree(), zorder=100
    )
    ax.add_patch(poly1)

    poly2 = patches.Polygon(
        coords_map['south'],
        closed=True, ec='red', fc='none', lw=2.5, ls='--',
        transform=ccrs.PlateCarree(), zorder=100
    )
    ax.add_patch(poly2)
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-121, ty+0.5, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=.8)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=.8)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=.8)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=(i==0); gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.025, 0.7])
cbar=fig.colorbar(cf, ticks=[-3,-2,-1,0,1,2,3], orientation='vertical', extend='both', cax=cax)
cbar.set_label('$\Delta$ Precipitation [mm/day]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

plt.savefig('../figs/prec_flor_pi_imerg-bias_change.pdf', transparent=False, bbox_inches='tight')
plt.savefig('../figs/prec_flor_pi_imerg-bias_change.png', transparent=False, bbox_inches='tight')

### Comparison of CTRL bias against IMERG, TRMM, GPCP, GPCC precip products

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['CTRL$-$IMERG', 'CTRL$-$TRMM', 'CTRL$-$GPCP', 'CTRL$-$GPCC'])
letters=['A','B','C','D']
tx=-110.5
ty=40.25
# var specs
lat=jas_mean['flor']['ctrl'].lat
lon=jas_mean['flor']['ctrl'].lon
# bias colormap
dcmap,_,_,_=get_settings(field='precip', diff=True)
dvmin=-3
dvmax=3
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(1000,3000,7)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[239,260,15,40]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(25,8), layout='constrained', subplot_kw={'projection':proj})

ax[0].pcolormesh(lon, lat, imerg_diff_mask['flor']['ctrl'], cmap=dcmap, norm=dnorm, transform=trans)
ax[1].pcolormesh(lon, lat, trmm_diff_mask['flor']['ctrl'], cmap=dcmap, norm=dnorm, transform=trans)
ax[2].pcolormesh(lon, lat, gpcp_diff_mask['flor']['ctrl'], cmap=dcmap, norm=dnorm, transform=trans)
cf=ax[3].pcolormesh(lon, lat, gpcc_diff_mask['flor']['ctrl'], cmap=dcmap, norm=dnorm, transform=trans)

for i, ax in enumerate(ax.flat): 
    # add topo contours
    ax.contour(lon, lat, topo['ctrl'], levels=zlevels, linewidths=1.3, colors='black', transform=trans)
    # add boxes around NAM sub-domains
    poly1 = patches.Polygon(
        coords_map['north'],
        closed=True, ec='firebrick', fc='none', lw=2.5, ls='--',
        transform=ccrs.PlateCarree(), zorder=100
    )
    ax.add_patch(poly1)

    poly2 = patches.Polygon(
        coords_map['south'],
        closed=True, ec='red', fc='none', lw=2.5, ls='--',
        transform=ccrs.PlateCarree(), zorder=100
    )
    ax.add_patch(poly2)
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-121, ty+0.5, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=.8)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=.8)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=.8)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=(i==0); gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}


# colorbar
cax=fig.add_axes([1.01, 0.15, 0.02, 0.7])
cbar=fig.colorbar(cf, ticks=[-3,-2,-1,0,1,2,3], orientation='vertical', extend='both', cax=cax)
cbar.set_label('Precipitation Bias [mm/day]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

#plt.savefig(f'{opath}/prec.bias.{season}.flor-ctrl.imerg.trmm.gpcp.gpcc.pdf', transparent=False, bbox_inches='tight')

### Plot against all precipitation products

In [ ]:
products = [
    dict(
        name='IMERG',
        diff=imerg_diff_mask,
        hatch=imerg_bias_change_masked
    ),
    dict(
        name='TRMM',
        diff=trmm_diff_mask,
        hatch=trmm_bias_change_masked
    ),
    dict(
        name='GPCP',
        diff=gpcp_diff_mask,
        hatch=gpcp_bias_change_masked
    ),
    dict(
        name='GPCC',
        diff=gpcc_diff_mask,
        hatch=gpcc_bias_change_masked
    ),
]

# ---------- Formatting settings ----------
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['CTRL$-$OBS', r'HI$_{\mathbf{gbl}}$$-$CTRL', r'HI$_{\mathbf{mex}}$$-$CTRL'])
letters = list("ABCDEFGHIJKLMN"); letter_i = 0
tx=-110.5
ty=40.25
# var specs
lat=jas_mean['flor']['ctrl'].lat
lon=jas_mean['flor']['ctrl'].lon
# bias colormap
dcmap,_,_,_=get_settings(field='precip', diff=True)
dvmin=-3
dvmax=3
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(1000,3000,7)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[239,260,15,40]

nrows = len(products)
ncols = 3

fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 5.5*nrows),
                       layout='constrained', subplot_kw={'projection': proj} )


for r, prod in enumerate(products):

    # ---------- Column 1: Model-Obs bias (product-specific) ----------
    # ctrl bias
    cf = ax[r,0].pcolormesh(lon, lat,
                            prod['diff'][key]['ctrl'],
                            cmap=dcmap, norm=dnorm, transform=trans)
    # topo contours
    ax[r,0].contour(lon, lat,
                    topo['ctrl'],
                    levels=zlevels, linewidths=1.3, colors='k', transform=trans)
    # prod name label on left
    ax[r,0].text(-125, 27, prod['name'], rotation=90, va='center', **text_kw)

    # ---------- Columns 2–3: model diffs
    for c, run in enumerate(['hitopo','hicam']):
        col=c+1
        # change in prec in flor mod topo experiments
        ax[r,col].pcolormesh(lon, lat,
                             model_diff_mask[key][run],
                             cmap=dcmap, norm=dnorm, transform=trans)
        # product-specific hatching
        ax[r,col].contourf(lon, lat,
                           prod['hatch'][key][run],
                           0, colors='none', hatches=['','///'], extend='lower', zorder=100, transform=trans)
        # topo contours
        ax[r,col].contour(lon, lat,
                          topo[run],
                          levels=zlevels, linewidths=1.3, colors='black', transform=trans)

    # ---------- Formatting ----------
    for c in range(ncols):
        ax[r,c].text(-121, ty+0.6, letters[letter_i], **text_kw2)
        letter_i+=1
        if r==0: # column header
            ax[r,c].text(tx, ty, titles[c], va='bottom', **text_kw)

for r in range(nrows):
    for c in range(ncols):
        a=ax[r,c]
        # add boxes around NAM sub-domains
        poly1 = patches.Polygon(
            coords_map['north'],
            closed=True, ec='firebrick', fc='none', lw=2.5, ls='--',
            transform=ccrs.PlateCarree(), zorder=100
        )
        a.add_patch(poly1)
        poly2 = patches.Polygon(
            coords_map['south'],
            closed=True, ec='red', fc='none', lw=2.5, ls='--',
            transform=ccrs.PlateCarree(), zorder=100
        )
        a.add_patch(poly2)
        a.coastlines(color='k', linewidth=0.8)
        a.add_feature(cfeature.STATES, edgecolor='k', linewidth=0.8)
        a.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=0.8)
        a.set_extent(map_bnds, crs=trans)
        gl = a.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.bottom_labels=(r == nrows - 1); gl.left_labels=(c == 0); gl.top_labels=False; gl.right_labels=False
        gl.xformatter = LONGITUDE_FORMATTER; gl.yformatter = LATITUDE_FORMATTER
        gl.xlabel_style = {'size':14}; gl.ylabel_style = {'size':14}

cax = fig.add_axes([.15, -0.021, 0.7, 0.0175])
cbar = fig.colorbar(cf, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cax)
cbar.set_label('$\Delta$ Precipitation [mm/day]', labelpad=5, size=18)
cbar.ax.tick_params(labelsize=18)

plt.savefig('../figs/prec_flor_pi_all-obs-prods-bias_change.pdf', transparent=False, bbox_inches='tight')
plt.savefig('../figs/prec_flor_pi_all-obs-prods-bias_change.png', transparent=False, bbox_inches='tight')